# Feature Exploration: Holiday Features (Multi-Route)

This notebook tests the impact of holiday features on the latest prediction model, which uses the multi-route data with low-volume filtering. Note that from the exploratory data analysis in notebook 3b, these features show weak individual correlations (<0.06), but had **not previously been evaluated** with the ML models. This notebook evaluates whether they provide predictive value despite weak correlations. The motivation came from the previous feature exploration, where adding a weather feature to the more recent multi-route model resulted in meaningful improvements.

**Holiday Features to Test (from 3b):**
- State public holidays: `n_public_holidays_{state}` for 8 states
- Aggregate: `n_public_holidays_national`, `n_public_holidays_total`
- Binary flag: `has_major_holiday` (Christmas, Easter, ANZAC)
- School holidays: `school_holiday_days`, `pct_school_holiday`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from calendar import monthrange
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# For holiday generation
import holidays

# Plotting style
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

# XGBoost for classification
try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation (same as 6d)

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute.csv')

# Parse dates
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)

# Create unique identifier for each airline-route combination
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']

# Sort for proper lag creation
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

## 2. Generate Holiday Features

In [ ]:
# Create holiday calendars for all Australian states and territories
years = list(range(2010, 2026))
STATES = ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']

state_holidays = {}
for state in STATES:
    state_holidays[state] = holidays.Australia(years=years, prov=state)

national_holidays = holidays.Australia(years=years)
print(f"Holiday calendars loaded for {len(STATES)} states + national")

In [ ]:
# Helper functions from notebook 3b
def count_holidays_in_month(holiday_dict, year, month):
    """Count number of public holidays in a given month."""
    count = 0
    _, days_in_month = monthrange(year, month)
    for day in range(1, days_in_month + 1):
        if date(year, month, day) in holiday_dict:
            count += 1
    return count

def has_major_holiday_in_month(holiday_dict, year, month):
    """Check if month contains major holidays (Christmas, Easter, ANZAC)."""
    major_keywords = ['christmas', 'easter', 'anzac', 'good friday', 'boxing']
    _, days_in_month = monthrange(year, month)
    for day in range(1, days_in_month + 1):
        d = date(year, month, day)
        if d in holiday_dict:
            name = holiday_dict[d].lower()
            if any(kw in name for kw in major_keywords):
                return 1
    return 0

def get_school_holiday_periods(year):
    """Return list of (start_date, end_date) tuples for school holidays."""
    periods = [
        (date(year-1, 12, 18), date(year, 1, 28)),  # Summer (Dec-Jan)
        (date(year, 4, 8), date(year, 4, 23)),       # Easter/Autumn
        (date(year, 6, 27), date(year, 7, 14)),      # Winter
        (date(year, 9, 23), date(year, 10, 7)),      # Spring
        (date(year, 12, 18), date(year, 12, 31)),    # Summer start
    ]
    return periods

def count_school_holiday_days_in_month(year, month):
    """Count days in a month that fall within school holiday periods."""
    periods = get_school_holiday_periods(year)
    if month == 1:
        periods.extend(get_school_holiday_periods(year))
    
    _, days_in_month = monthrange(year, month)
    holiday_days = 0
    
    for day in range(1, days_in_month + 1):
        current_date = date(year, month, day)
        for start, end in periods:
            if start <= current_date <= end:
                holiday_days += 1
                break
    return holiday_days

In [ ]:
# Get unique year-months from data
year_months = df[['year', 'month_num']].drop_duplicates().sort_values(['year', 'month_num'])
print(f"Generating holiday features for {len(year_months)} unique months...")

# Generate all holiday features
holiday_features = []

for _, row in year_months.iterrows():
    year, month = int(row['year']), int(row['month_num'])
    _, days_in_month = monthrange(year, month)
    
    feature_row = {'year': year, 'month_num': month}
    
    # Public holidays by state
    all_holiday_dates = set()
    for state in STATES:
        n_state = count_holidays_in_month(state_holidays[state], year, month)
        feature_row[f'n_public_holidays_{state.lower()}'] = n_state
        
        for day in range(1, days_in_month + 1):
            d = date(year, month, day)
            if d in state_holidays[state]:
                all_holiday_dates.add(d)
    
    # National holidays
    feature_row['n_public_holidays_national'] = count_holidays_in_month(national_holidays, year, month)
    
    # Total unique holidays
    feature_row['n_public_holidays_total'] = len(all_holiday_dates)
    
    # Major holiday flag
    has_major = 0
    for state in STATES:
        if has_major_holiday_in_month(state_holidays[state], year, month):
            has_major = 1
            break
    feature_row['has_major_holiday'] = has_major
    
    # School holidays
    school_days = count_school_holiday_days_in_month(year, month)
    feature_row['school_holiday_days'] = school_days
    feature_row['pct_school_holiday'] = round(school_days / days_in_month, 4)
    
    holiday_features.append(feature_row)

df_holidays = pd.DataFrame(holiday_features)
print(f"Holiday features shape: {df_holidays.shape}")
print(f"Columns: {list(df_holidays.columns)}")

In [ ]:
# Merge holiday features into main dataframe
df = df.merge(df_holidays, on=['year', 'month_num'], how='left')
print(f"Shape after merge: {df.shape}")

# Check for missing values
holiday_cols = [c for c in df_holidays.columns if c not in ['year', 'month_num']]
print(f"Missing values in holiday features: {df[holiday_cols].isnull().sum().sum()}")

## 3. Filter Low-Volume Airline-Routes (from 6d)

In [ ]:
# Calculate average volume per airline-route
volume_threshold = 50  # From notebook 6b

airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']

# Identify high-volume airline-routes
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()

# Filter to high-volume airline-routes only
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

print(f"Volume threshold: {volume_threshold} flights/month")
print(f"\nRecords before filtering: {len(df)}")
print(f"Records after filtering:  {len(df_filtered)}")
print(f"Remaining airline-routes: {df_filtered['airline_route'].nunique()}")

## 4. Explore Holiday Feature Correlations

In [ ]:
# Check correlations with delay_rate
print("Holiday Feature Correlations with delay_rate:")
print("=" * 50)

print("\nBy State:")
for state in STATES:
    col = f'n_public_holidays_{state.lower()}'
    corr = df_filtered[col].corr(df_filtered['delay_rate'])
    print(f"  {col:<30} {corr:>8.4f}")

print("\nAggregate Features:")
for col in ['n_public_holidays_national', 'n_public_holidays_total', 'has_major_holiday',
            'school_holiday_days', 'pct_school_holiday']:
    corr = df_filtered[col].corr(df_filtered['delay_rate'])
    print(f"  {col:<30} {corr:>8.4f}")

In [ ]:
# Correlation by route
print("\nHoliday Correlations by Route:")
print("=" * 70)
print(f"{'Route':<20} {'major_holiday':>14} {'school_days':>14} {'total_hols':>14}")
print("-" * 70)

for route in sorted(df_filtered['route'].unique()):
    route_data = df_filtered[df_filtered['route'] == route]
    corr_major = route_data['has_major_holiday'].corr(route_data['delay_rate'])
    corr_school = route_data['school_holiday_days'].corr(route_data['delay_rate'])
    corr_total = route_data['n_public_holidays_total'].corr(route_data['delay_rate'])
    print(f"{route:<20} {corr_major:>14.4f} {corr_school:>14.4f} {corr_total:>14.4f}")

In [ ]:
# Visualize holiday impact
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Delay rate by major holiday flag
ax = axes[0]
grouped = df_filtered.groupby('has_major_holiday')['delay_rate'].agg(['mean', 'std'])
ax.bar([0, 1], grouped['mean'], yerr=grouped['std'], capsize=5, color=['steelblue', 'coral'], alpha=0.8)
ax.set_xticks([0, 1])
ax.set_xticklabels(['No Major Holiday', 'Major Holiday'])
ax.set_ylabel('Mean Delay Rate')
ax.set_title('Delay Rate by Major Holiday')
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Delay rate vs school holiday days
ax = axes[1]
ax.scatter(df_filtered['school_holiday_days'], df_filtered['delay_rate'], alpha=0.3, s=10)
ax.set_xlabel('School Holiday Days')
ax.set_ylabel('Delay Rate')
ax.set_title('Delay Rate vs School Holiday Days')
ax.grid(True, alpha=0.3)

# Plot 3: Delay rate vs total public holidays
ax = axes[2]
ax.scatter(df_filtered['n_public_holidays_total'], df_filtered['delay_rate'], alpha=0.3, s=10)
ax.set_xlabel('Total Public Holidays')
ax.set_ylabel('Delay Rate')
ax.set_title('Delay Rate vs Total Public Holidays')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Create lagged features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)

# Create momentum feature
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Create cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encode airline
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

# One-hot encode route
route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Create weather features (from 6d baseline)
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())

# Drop rows with missing lag values
df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"Rows after dropping NaN: {len(df_clean)}")

## 6. Train/Validation/Test Split

In [ ]:
# Time-based split (excluding 2020-2022 COVID period)
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2023))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")

In [ ]:
# Define feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']

# Baseline features (6d without wind)
baseline_features = base_features + ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp']

# Holiday feature groups to test
state_holiday_cols = [f'n_public_holidays_{s.lower()}' for s in STATES]
aggregate_holiday_cols = ['n_public_holidays_national', 'n_public_holidays_total', 'has_major_holiday']
school_holiday_cols = ['school_holiday_days', 'pct_school_holiday']

# Feature variants to test
holiday_variants = {
    'baseline': baseline_features,
    '+aggregate_holidays': baseline_features + aggregate_holiday_cols,
    '+school_holidays': baseline_features + school_holiday_cols,
    '+all_holidays': baseline_features + aggregate_holiday_cols + school_holiday_cols,
    '+state_holidays': baseline_features + state_holiday_cols,
}

print("Feature variants:")
for name, features in holiday_variants.items():
    print(f"  {name}: {len(features)} features")

In [ ]:
# Prepare target variables
y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

print("Target variables prepared.")

## 7. Regression Models: Baseline vs +Holidays

In [ ]:
# Train regression models for each variant
reg_results = []

for variant_name, features in holiday_variants.items():
    print(f"\n{'='*60}")
    print(f"Regression with: {variant_name}")
    print(f"{'='*60}")
    
    # Prepare data
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    # Scale for linear models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Ridge Regression
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train_reg)
    ridge_test_pred = ridge.predict(X_test_scaled)
    
    ridge_r2 = r2_score(y_test_reg, ridge_test_pred)
    ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
    print(f"  Ridge:    R²={ridge_r2:.4f}, RMSE={ridge_rmse:.4f}")
    
    reg_results.append({
        'Variant': variant_name,
        'Model': 'Ridge',
        'Test_R2': ridge_r2,
        'Test_RMSE': ridge_rmse
    })
    
    # Random Forest Regression
    rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_reg.fit(X_train, y_train_reg)
    rf_test_pred = rf_reg.predict(X_test)
    
    rf_r2 = r2_score(y_test_reg, rf_test_pred)
    rf_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
    print(f"  RF:       R²={rf_r2:.4f}, RMSE={rf_rmse:.4f}")
    
    reg_results.append({
        'Variant': variant_name,
        'Model': 'Random Forest',
        'Test_R2': rf_r2,
        'Test_RMSE': rf_rmse
    })

reg_df = pd.DataFrame(reg_results)

## 8. Classification Models: Baseline vs +Holidays

In [ ]:
# Train classification models for each variant
clf_results = []

# Simpler variant set for classification (skip state holidays to save time)
clf_variants = {
    'baseline': baseline_features,
    '+aggregate_holidays': baseline_features + aggregate_holiday_cols,
    '+school_holidays': baseline_features + school_holiday_cols,
    '+all_holidays': baseline_features + aggregate_holiday_cols + school_holiday_cols,
}

for variant_name, features in clf_variants.items():
    print(f"\n{'='*60}")
    print(f"Classification with: {variant_name}")
    print(f"{'='*60}")
    
    # Prepare data
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    # Scale for linear models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Logistic Regression
    logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train_clf)
    logreg_test_pred = logreg.predict(X_test_scaled)
    logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]
    
    logreg_f1 = f1_score(y_test_clf, logreg_test_pred)
    logreg_auc = roc_auc_score(y_test_clf, logreg_test_proba)
    print(f"  Logistic: F1={logreg_f1:.4f}, AUC={logreg_auc:.4f}")
    
    clf_results.append({
        'Variant': variant_name,
        'Model': 'Logistic',
        'Test_F1': logreg_f1,
        'Test_AUC': logreg_auc
    })
    
    # Random Forest Classification
    rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train, y_train_clf)
    rf_clf_test_pred = rf_clf.predict(X_test)
    rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]
    
    rf_clf_f1 = f1_score(y_test_clf, rf_clf_test_pred)
    rf_clf_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
    print(f"  RF Clf:   F1={rf_clf_f1:.4f}, AUC={rf_clf_auc:.4f}")
    
    clf_results.append({
        'Variant': variant_name,
        'Model': 'Random Forest',
        'Test_F1': rf_clf_f1,
        'Test_AUC': rf_clf_auc
    })
    
    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            min_child_weight=5, random_state=42, n_jobs=-1
        )
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_test_pred = xgb_clf.predict(X_test)
        xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
        
        xgb_f1 = f1_score(y_test_clf, xgb_test_pred)
        xgb_auc = roc_auc_score(y_test_clf, xgb_test_proba)
        print(f"  XGBoost:  F1={xgb_f1:.4f}, AUC={xgb_auc:.4f}")
        
        clf_results.append({
            'Variant': variant_name,
            'Model': 'XGBoost',
            'Test_F1': xgb_f1,
            'Test_AUC': xgb_auc
        })

clf_df = pd.DataFrame(clf_results)

## 9. Side-by-Side Comparison

In [ ]:
# Regression comparison
print("=" * 100)
print("REGRESSION: Baseline vs Holiday Variants")
print("=" * 100)

reg_pivot = reg_df.pivot(index='Model', columns='Variant', values='Test_R2')
reg_pivot = reg_pivot[['baseline', '+aggregate_holidays', '+school_holidays', '+all_holidays', '+state_holidays']]

print(f"\n{'Model':<15} {'baseline':>12} {'+aggregate':>12} {'+school':>12} {'+all':>12} {'+state':>12}")
print("-" * 80)
for model in reg_pivot.index:
    row = reg_pivot.loc[model]
    print(f"{model:<15} {row['baseline']:>12.4f} {row['+aggregate_holidays']:>12.4f} {row['+school_holidays']:>12.4f} {row['+all_holidays']:>12.4f} {row['+state_holidays']:>12.4f}")

In [ ]:
# Classification comparison
print("=" * 100)
print("CLASSIFICATION: Baseline vs Holiday Variants")
print("=" * 100)

clf_pivot = clf_df.pivot(index='Model', columns='Variant', values='Test_F1')
clf_pivot = clf_pivot[['baseline', '+aggregate_holidays', '+school_holidays', '+all_holidays']]

print(f"\n{'Model':<15} {'baseline':>12} {'+aggregate':>12} {'+school':>12} {'+all':>12}")
print("-" * 70)
for model in clf_pivot.index:
    row = clf_pivot.loc[model]
    print(f"{model:<15} {row['baseline']:>12.4f} {row['+aggregate_holidays']:>12.4f} {row['+school_holidays']:>12.4f} {row['+all_holidays']:>12.4f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette("Set2", 5)

# Regression R²
ax = axes[0]
x_reg = np.arange(len(reg_pivot.index))
width = 0.15
variants = ['baseline', '+aggregate_holidays', '+school_holidays', '+all_holidays', '+state_holidays']
colors = palette

for i, variant in enumerate(variants):
    ax.bar(x_reg + i*width, reg_pivot[variant], width, label=variant, color=colors[i], alpha=0.8)

ax.set_ylabel(r'Test $R^2$')
ax.set_title(r'Regression: $R^2$ Comparison')
ax.set_xticks(x_reg + width * 2)
ax.set_xticklabels(reg_pivot.index)
ax.legend(fontsize=8)
ax.set_ylim(0, 0.6)
ax.grid(True, alpha=0.3, axis='y')

# Classification F1
ax = axes[1]
x_clf = np.arange(len(clf_pivot.index))  # Use separate x for classification
clf_variants_plot = ['baseline', '+aggregate_holidays', '+school_holidays', '+all_holidays']

for i, variant in enumerate(clf_variants_plot):
    ax.bar(x_clf + i*width, clf_pivot[variant], width, label=variant, color=colors[i], alpha=0.8)

ax.set_ylabel(r'Test $F_1$')
ax.set_title(r'Classification: $F_1$ Comparison')
ax.set_xticks(x_clf + width * 1.5)
ax.set_xticklabels(clf_pivot.index)
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Feature Importance Analysis

In [ ]:
# Train RF with all holidays to check feature importance
all_holiday_features = baseline_features + aggregate_holiday_cols + school_holiday_cols
X_train_reg = df_clean.loc[train_mask, all_holiday_features].values

rf_final = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_final.fit(X_train_reg, y_train_reg)

# Feature importance
importance_df = pd.DataFrame({
    'Feature': all_holiday_features,
    'Importance': rf_final.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (RF Regression with all holidays):")
print("-" * 60)
for _, row in importance_df.head(20).iterrows():
    marker = " <- HOLIDAY" if any(h in row['Feature'] for h in ['holiday', 'school']) else ""
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}{marker}")

# Show holiday features specifically
print("\nHoliday Feature Importance:")
print("-" * 60)
holiday_importance = importance_df[importance_df['Feature'].str.contains('holiday|school')]
for _, row in holiday_importance.iterrows():
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}")

## 11. Summary and Observations

In [ ]:
print("="*80)
print("SUMMARY: Impact of Holiday Features on Multi-Route Model")
print("="*80)

# Calculate improvements from baseline
baseline_ridge_r2 = reg_pivot.loc['Ridge', 'baseline']
baseline_rf_r2 = reg_pivot.loc['Random Forest', 'baseline']
baseline_log_f1 = clf_pivot.loc['Logistic', 'baseline']
baseline_rf_clf_f1 = clf_pivot.loc['Random Forest', 'baseline']
baseline_xgb_f1 = clf_pivot.loc['XGBoost', 'baseline'] if 'XGBoost' in clf_pivot.index else None

# Summary metrics table - Regression
print("\nREGRESSION PERFORMANCE:")
print("-" * 80)
print(f"{'Model':<15} {'Baseline':>12} {'Best +Holiday':>14} {'Δ R²':>10} {'Best Variant':>20}")
print("-" * 80)

for model in ['Ridge', 'Random Forest']:
    baseline = reg_pivot.loc[model, 'baseline']
    best_variant = reg_pivot.loc[model, ['+aggregate_holidays', '+school_holidays', '+all_holidays', '+state_holidays']].idxmax()
    best_val = reg_pivot.loc[model, best_variant]
    diff = best_val - baseline
    sign = '+' if diff > 0 else ''
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<15} {baseline:>12.4f} {best_val:>14.4f} {sign}{diff:>9.4f} {best_variant:>20} ({status})")

# Summary metrics table - Classification
print("\nCLASSIFICATION PERFORMANCE:")
print("-" * 80)
print(f"{'Model':<15} {'Baseline':>12} {'Best +Holiday':>14} {'Δ F1':>10} {'Best Variant':>20}")
print("-" * 80)

for model in clf_pivot.index:
    baseline = clf_pivot.loc[model, 'baseline']
    best_variant = clf_pivot.loc[model, ['+aggregate_holidays', '+school_holidays', '+all_holidays']].idxmax()
    best_val = clf_pivot.loc[model, best_variant]
    diff = best_val - baseline
    sign = '+' if diff > 0 else ''
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<15} {baseline:>12.4f} {best_val:>14.4f} {sign}{diff:>9.4f} {best_variant:>20} ({status})")

# Count improvements
reg_improvements = sum(1 for m in ['Ridge', 'Random Forest'] 
                       if reg_pivot.loc[m, ['+aggregate_holidays', '+school_holidays', '+all_holidays', '+state_holidays']].max() - reg_pivot.loc[m, 'baseline'] > 0.005)
clf_improvements = sum(1 for m in clf_pivot.index 
                       if clf_pivot.loc[m, ['+aggregate_holidays', '+school_holidays', '+all_holidays']].max() - clf_pivot.loc[m, 'baseline'] > 0.005)

total_models = 2 + len(clf_pivot.index)
total_improvements = reg_improvements + clf_improvements

### Observations:


- The correlations of the holiday features remain low, confirming the findings from notebook 3b
- However, they do help nudge the predictive performance higher, especially for the classification models.


## 12. Next step

The decision to include holiday features is deferred. Only 2–3 features with the highest importance will be retained, as some features contain repeated information. For example, `pct_school_holiday` and `school_holiday_days`.

Next, the effect of route distance will be investigated. This comes from the hypothesis that short routes may have higher delay rate, as there is less margin of error.

See [7b_feature_exploration_route_distance.ipynb](/notebooks/7b_feature_exploration_route_distance.ipynb).
